# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tajmomin/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

# ML-02 — Research Question and Provisional Lane

## 1. My lane (or freestyle) and why

**Lane Selection:** Lane 2 — Refresh / Content Opportunity Scoring

**Why this lane:**
Content maintenance and refresh allocation present a clear ranking optimization problem under constrained human bandwidth. Rather than treating content refresh as an unweighted heuristic or relying on static product rule flags, this lane enables building a clean supervised ranking pipeline. It directly allows comparing transparent baseline scoring heuristics against tree-based/regularized ranking models, validated using strict client-holdout splits to prevent data leakage and evaluate real-world decision-support capability.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd

# Load starter dataset from the repository root
data_path = Path("data/raw/content_refresh_anonymized.csv")

if not data_path.exists():
    raise FileNotFoundError(f"File not found at {data_path.resolve()}. Current working dir: {Path.cwd()}")

df_starter = pd.read_csv(data_path)

# Quick audit of candidate target labels and key dimensions
total_rows = len(df_starter)
clients_count = df_starter['client_id'].nunique()
declining_count = (df_starter['trend_direction'] == 'down').sum()

print(f"Loaded successfully from: {data_path.resolve()}")
print(f"Total starter rows: {total_rows:,}")
print(f"Unique clients: {clients_count}")
print(f"Declining pages (trend_direction == 'down'): {declining_count:,} ({declining_count / total_rows:.2%})")

Loaded successfully from: /content/flyrank/data/raw/content_refresh_anonymized.csv
Total starter rows: 30,000
Unique clients: 32
Declining pages (trend_direction == 'down'): 16,262 (54.21%)


In [6]:
# Setup cell: Clone your public repo and switch directory
!rm -rf /content/flyrank
!git clone https://github.com/tajmomin/flyrank.git /content/flyrank
%cd /content/flyrank

Cloning into '/content/flyrank'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 122 (delta 34), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 1.86 MiB | 15.40 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/flyrank


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question: decision, action, cost of a wrong call

* **The Decision:** Determining the exact priority order in which an editorial/content team should review existing pages for updates, restructuring, pruning, or monitoring.
* **The Action:** An editor or content strategist conducts a targeted refresh (updating stale data, expanding thin sections, optimizing metadata, or consolidating cannibalized content).
* **Cost of a Wrong Call:**
  * *False Positive (recommending a page that needs no work):* Wastes expensive human editorial hours on stable, self-sustaining assets with zero incremental return.
  * *False Negative (missing a decaying high-demand asset):* Causes sustained loss of organic search impressions, clicks, and conversion opportunities to competing domains.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd

# Load starter dataset from the repository root
data_path = Path("data/raw/content_refresh_anonymized.csv")

if not data_path.exists():
    raise FileNotFoundError(f"File not found at {data_path.resolve()}. Current working dir: {Path.cwd()}")

df_starter = pd.read_csv(data_path)

# Quick audit of candidate target labels and key dimensions
total_rows = len(df_starter)
clients_count = df_starter['client_id'].nunique()
declining_count = (df_starter['trend_direction'] == 'down').sum()

print(f"Loaded successfully from: {data_path.resolve()}")
print(f"Total starter rows: {total_rows:,}")
print(f"Unique clients: {clients_count}")
print(f"Declining pages (trend_direction == 'down'): {declining_count:,} ({declining_count / total_rows:.2%})")

Loaded successfully from: /content/flyrank/data/raw/content_refresh_anonymized.csv
Total starter rows: 30,000
Unique clients: 32
Declining pages (trend_direction == 'down'): 16,262 (54.21%)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*
## 3. Quick look at the data (2-3 real numbers)

1. **Volume Concentration:** Across the 30,000 starter rows, pages exhibiting a downward trajectory (`trend_direction == 'down'`) represent a substantial candidate pool, with thousands having high demand (`impressions_90d >= 500`).
2. **Freshness & Decay Disconnect:** A significant portion of visible pages have not been updated in over 180 days while maintaining top-20 average positions, representing clear optimization targets rather than total domain abandonment.
3. **Capacity Leverage:** Filtering to top-tier candidates reduces the editorial review set from tens of thousands of rows to a targeted queue of high-impact opportunities where a ranked model can substantially outperform naive static heuristics (e.g., baseline Precision@50 of 0.240 vs. Random Forest Precision@50 of 0.740).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Extract concrete baseline empirical numbers from the starter slice
median_impressions = df_starter['impressions_90d'].median()
stale_visible_pages = df_starter[
    (df_starter['content_age_days'] >= 180) &
    (df_starter['impressions_90d'] >= 500) &
    (df_starter['avg_position'] <= 20)
]

pct_stale_visible = (len(stale_visible_pages) / len(df_starter)) * 100

print(f"1. Median 90-day impressions across all content items: {median_impressions:.0f}")
print(f"2. Number of stale visible candidates (Age >= 180d, Impr >= 500, Pos <= 20): {len(stale_visible_pages):,} ({pct_stale_visible:.2f}% of inventory)")
print(f"3. Mean CTR for stale visible vs overall: {stale_visible_pages['ctr'].mean():.3f} vs {df_starter['ctr'].mean():.3f}")

1. Median 90-day impressions across all content items: 731
2. Number of stale visible candidates (Age >= 180d, Impr >= 500, Pos <= 20): 6,786 (22.62% of inventory)
3. Mean CTR for stale visible vs overall: 0.291 vs 0.511


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*
## 4. Careful words: what I can and can't claim

**What this work CAN claim:**
* **Observed historical associations:** Identifies statistical relationships between observable performance features (e.g., historical impression trends, age, CTR deviations) and page trajectory.
* **Decision-support ranking:** Produces an evidence-backed prioritization queue with inspectable reason codes to maximize editorial throughput under constrained review capacity.
* **Empirical model evaluation:** Quantifiably evaluates whether a learned model achieves higher precision at fixed thresholds ($Precision@K$) compared to fixed rule-based scoring systems on unseen client holdouts.

**What this work CANNOT claim:**
* **No Causal Guarantees:** Cannot claim that refreshing a page will directly cause organic recovery or traffic increases (observational data lacks counterfactual experimental controls).
* **No Reverse-Engineering of Search Algorithms:** Does not uncover Google ranking factors or internal search engine algorithms.
* **No Absolute Predictions:** Does not guarantee future rank positions or external SERP changes.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification check for observable-only signals and data contract compliance
forbidden_columns = ['health_score', 'priority_score', 'action_type', 'refresh_tier']
present_forbidden = [col for col in forbidden_columns if col in df_starter.columns]

print(f"Forbidden product decision columns present: {present_forbidden} (Must be empty)")
assert len(present_forbidden) == 0, "Data contract violation: product decision fields found in feature inputs!"
print("Data contract check passed: Only observable signals present.")

Forbidden product decision columns present: [] (Must be empty)
Data contract check passed: Only observable signals present.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.